# Neural network classifier (from scratch)

A neural network (multi-layer perceptron) extends the linear classifier by stacking an affine transform, a non-linear activation (ReLU), and a second affine transform before the softmax. This lets it learn non-linear decision boundaries — something a single linear classifier can't do. This notebook implements the forward pass, the softmax cross-entropy loss, and backpropagation from scratch (vectorized), training with gradient descent — see "Neural Network Classifier with sklearn" for the scikit-learn version.

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/04_neural-networks'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

In [ ]:
![ -f utils.py ] || gdown --id 1NUGtNIABwt4PSAkBXPcMtnKPPaZYmxP3 -O utils.py

## Artificial dataset

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from utils import standard_scaler

num_classes = 2
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)  # try increasing the noise to see what happens

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize X (try to comment this out and see what happens)
X_train, mean, std = standard_scaler(X_train)
X_test, _, _ = standard_scaler(X_test, mean, std)

print(f"Train set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# Plot the dataset
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors="gray")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")

print(f"X_train_min: {X_train.min()}, X_train_max: {X_train.max()}")
print(f"ymin: {y.min()}, ymax: {y.max()}")

## Model

In [ ]:
import numpy as np


def affine_forward(X, W, b):
    """
    Computes the forward pass for an affine (fully connected) layer.

    Inputs:
    - X: Input data, of shape (N, D)
    - W: Weights, of shape (D, H)
    - b: Biases, of shape (H,)

    Returns a tuple of:
    - out: Output, of shape (N, H)
    - cache: (X, W, b) to be used in the backward pass
    """
    out = X @ W + b
    cache = (X, W, b)
    return out, cache


def affine_backward(grad_out, cache):
    """
    Computes the backward pass for an affine layer.

    Inputs:
    - grad_out: Upstream gradient, of shape (N, H)
    - cache: Tuple (X, W, b) from the forward pass

    Returns a tuple of:
    - grad_X: Gradient with respect to X, of shape (N, D)
    - grad_W: Gradient with respect to W, of shape (D, H)
    - grad_b: Gradient with respect to b, of shape (H,)
    """
    X, W, b = cache
    grad_X = grad_out @ W.T
    grad_W = X.T @ grad_out
    grad_b = grad_out.sum(axis=0)
    return grad_X, grad_W, grad_b


def relu_forward(X):
    """
    Computes the forward pass for a layer of rectified linear units (ReLUs).

    Inputs:
    - X: Input, of any shape

    Returns a tuple of:
    - out: Output, of the same shape as X
    - cache: X, to be used in the backward pass
    """
    out = np.maximum(0, X)
    cache = X
    return out, cache


def relu_backward(grad_out, cache):
    """
    Computes the backward pass for a layer of rectified linear units (ReLUs).

    Inputs:
    - grad_out: Upstream gradient, of any shape
    - cache: X, of the same shape as grad_out, from the forward pass

    Returns:
    - grad_X: Gradient with respect to X
    """
    X = cache
    grad_X = grad_out * (X > 0)
    return grad_X


def softmax_cross_entropy_loss(scores, y):
    """
    Computes the softmax loss (cross-entropy) and its gradient w.r.t. scores.

    Inputs:
    - scores: Input data, of shape (N, C) where x[i, j] is the score for the jth class
      for the ith input.
    - y: True labels, of shape (N,) where y[i] is the label for x[i].

    Returns a tuple of:
    - loss: Scalar giving the loss
    - grad: Gradient of the loss with respect to scores, of shape (N, C)
    """
    N = scores.shape[0]

    # Numerically stable softmax
    shifted_scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted_scores)
    probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)

    loss = -np.log(probs[np.arange(N), y]).mean()

    grad = probs.copy()
    grad[np.arange(N), y] -= 1
    grad /= N

    return loss, grad

## Training

In [ ]:
num_epochs = 1000  # try reducing this to 100 or increasing it to 5000
learning_rate = 1e-0  # try reducing this to 1e-1 or 1e-2
hidden_size = 32  # try reducing this to 4 to see underfitting
num_features = X_train.shape[1]
best_loss = float("inf")
best_params = None
loss_history = []

# Parameter initialization
W1 = np.random.randn(num_features, hidden_size) * 0.01
b1 = np.zeros(hidden_size)
W2 = np.random.randn(hidden_size, num_classes) * 0.01
b2 = np.zeros(num_classes)

# Training loop
for epoch in range(num_epochs):
    # Forward pass
    hidden, cache_affine1 = affine_forward(X_train, W1, b1)
    hidden_relu, cache_relu = relu_forward(hidden)
    scores, cache_affine2 = affine_forward(hidden_relu, W2, b2)

    # Compute loss and gradient w.r.t. scores
    loss, grad_scores = softmax_cross_entropy_loss(scores, y_train)

    # Backward pass
    grad_hidden_relu, grad_W2, grad_b2 = affine_backward(grad_scores, cache_affine2)
    grad_hidden = relu_backward(grad_hidden_relu, cache_relu)
    _, grad_W1, grad_b1 = affine_backward(grad_hidden, cache_affine1)

    # Update weights and biases
    W1 -= learning_rate * grad_W1
    b1 -= learning_rate * grad_b1
    W2 -= learning_rate * grad_W2
    b2 -= learning_rate * grad_b2

    # Store loss history
    loss_history.append(loss)

    # Check for best loss
    if loss < best_loss:
        best_loss = loss
        best_params = (W1.copy(), b1.copy(), W2.copy(), b2.copy())
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss:.4f}, Best Loss: {best_loss:.4f}")

In [ ]:
plt.plot(loss_history, label="Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss History")
plt.show()

## Test

In [ ]:
def predict(X, params):
    W1, b1, W2, b2 = params
    hidden, _ = affine_forward(X, W1, b1)
    hidden_relu, _ = relu_forward(hidden)
    scores, _ = affine_forward(hidden_relu, W2, b2)
    return np.argmax(scores, axis=1)


y_pred = predict(X_test, best_params)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[class_id for class_id in range(num_classes)])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[class_id for class_id in range(num_classes)])
disp.plot()
plt.show()

# Compute classification report
class_report = classification_report(y_test, y_pred, target_names=[str(class_id) for class_id in range(num_classes)])

print("\nClassification Report:")
print(class_report)

In [ ]:
def plot_decision_boundary(X, y, predict_fn, ax=None):
    if ax is None:
        ax = plt.gca()

    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200), np.linspace(x2_min, x2_max, 200))
    grid_preds = predict_fn(np.c_[xx1.ravel(), xx2.ravel()])
    grid_preds = grid_preds.reshape(xx1.shape)

    ax.contourf(xx1, xx2, grid_preds, alpha=0.2, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors="gray", cmap="coolwarm")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")


plot_decision_boundary(X_test, y_test, lambda X: predict(X, best_params))
plt.title("Decision boundary (test set)")
plt.show()

accuracy = np.sum(y_pred == y_test) / len(y_test)
print("Accuracy:", accuracy)